# MODIS over the Ogallala aquifer on MODIS's own grid: monthly vegetation indices (MOD13A3) and yearly land cover (MCD12Q1), one combined NetCDF file each

This notebook reads **MOD13A3 version 6.1** (Terra, monthly NDVI / EVI / reflectance / quality, 1 km) and **MCD12Q1 version 6.1**
(Terra + Aqua, yearly land cover, 500 m) from Google Earth Engine as the stored integers, and writes **one CF NetCDF file per dataset
for the whole record** to Drive. It replaces the earlier version of this notebook, which wrote one file per year and had to be merged by
a separate step.

| | MOD13A3 | MCD12Q1 |
|---|---|---|
| Output | `MyDrive/MSUGWB/MOD13A3/MOD13A3_v061_1km_Ogallala.nc` | `MyDrive/MSUGWB/MCD12Q1/MCD12Q1_v061_500m_Ogallala.nc` |
| Source | `MODIS/061/MOD13A3`, Feb 2000 → the latest month Earth Engine has | `MODIS/061/MCD12Q1`, 2001 → the latest year Earth Engine has |
| One record | one month | one year |
| Variables | `NDVI`, `EVI`, `sur_refl_b01/02/03/07`, `DetailedQA`, `SummaryQA`, `ViewZenith`, `SolarZenith`, `RelativeAzimuth` (11, every band Earth Engine has) | `LC_Type1-5`, `LC_Prop1-3`, `LC_Prop1-3_Assessment`, `QC`, `LW` (13, every band) |
| Grid | MODIS's sinusoidal grid, 926.6 m, **not resampled** | the same grid, 463.3 m, **not resampled** |
| Window | 1431 × 1871 cells, 530,113 inside the aquifer | 2861 × 3743 cells, 2,120,348 inside the aquifer |

The two products are on different grids (1 km and 500 m), so they are two files by design, not one.

**How to run**

1. *Run all* with `TEST_MODE = True`: four records of each dataset, spread over the record, into `.../_test/`, with every check switched
   on and a projection of the full run (from the rehearsal: about 8 s and 9 MB per MOD13A3 record, so roughly 45 min and 3 GB for
   the 320 months; MCD12Q1 is much smaller).
2. Set `TEST_MODE = False` and *Run all*. If the runtime dies or you stop it, run all again: the file continues from what is on Drive.
   Later, when Earth Engine has a new month, run all again and the new records are added.

**What the values are**

* **MODIS's numbers, untouched.** Each variable is one band of the Earth Engine image, stored in the integer type Earth Engine lists for it
  (`int16`, `uint16`, `int8`, `uint8`). Nothing is scaled, resampled, quality-screened or gap-filled. `scale_factor` is written as a CF
  attribute, so xarray and other readers decode to physical units on their own.
* **No data.** Each band keeps the product's own no-data value as `_FillValue`, **read from NASA's HDF files** (MOD13A3.A2020183.h09v05 and
  MCD12Q1.A2020001.h10v05, checked on 2026-10-22): −3000 for NDVI / EVI, **−1000 for the four reflectance bands** (the earlier notebook
  declared −3000 for them), −10000 for the zenith angles, −4000 for the relative azimuth (inside its own valid range, so a true −40.00° cannot
  be told from fill), 65535 for `DetailedQA`, −1 for `SummaryQA`, 255 for every MCD12Q1 band. NASA's `valid_range` is stored too.
* **Cells outside the polygon** hold the same fill value, and `roi_mask` marks the cells inside (cell centre inside the polygon). They are
  never written, so they take no space. `KEEP_WHOLE_WINDOW = True` keeps every cell of the window instead.
* **Time.** `time` is the first day of the month (year); `time_bnds` is the calendar month (year). Earth Engine's own end time is 30 (365)
  days after the start for every image and is not used. MOD13A3 has no January 2000.
* **Coordinates.** `x`, `y` (metres, sinusoidal sphere of radius 6371007.181 m, which is what Earth Engine's pixel positions follow) and the 2-D `lat`, `lon`
  of every cell centre. `crs` carries the projection (CF-1.9, WKT and GDAL `GeoTransform`).
* **Classes.** MCD12Q1's class tables come from Earth Engine's public catalogue entry and are written as `flag_values`, `flag_meanings`
  and `class_descriptions`. If that page cannot be read the attributes are left out; the data are not affected.
* **Provider quirks are kept, not hidden.** The last cell reports, per variable, how many cells lie at the fill and how many outside NASA's
  `valid_range` (for example a few `sur_refl_b07` cells slightly below 0).

**How it works** (one record at a time: fetch → check → write → read back → commit)

* The image list comes from Earth Engine's catalogue. The window is cut once from the product's pixel lattice: the rectangle of cells whose
  centre lies inside the aquifer polygon. MOD13A3 has images with 18000, 16800 and 15600 rows (cut at different latitudes); each is laid on
  the window by a shift of whole cells.
* A record's pixels come as the stored integers through `getPixels`, in blocks, up to 32 requests at a time, on Earth Engine's
  high-volume endpoint. Nothing is computed on Earth Engine. Earth Engine refuses a request above 48 MiB; a refused block is fetched in
  halves instead of failing. While one record is written, the next is already downloading.
* The file is built on Colab's scratch disk. Every block is checked when it arrives (bands, shape, integer type as the catalogue lists it)
  and compared with a checksum when read back from scratch; every record is read back from the file before it counts.
* Every hour, and when the run ends or is stopped, the closed file is copied to Drive with a size and MD5 comparison and renamed over the
  previous copy. A runtime that dies loses at most the last hour.

**What the notebook checks**

* **Before any bulk work**, windows of the download are compared with Earth Engine's own evaluation of the same pixels (random aquifer
  cells, and cells where the image holds a no-data value, because those are rare): values must be identical, and every cell Earth Engine
  masks must have arrived as the band's fill. This is the check that finds a wrong fill value.
* A band whose fill is not read from NASA's file (an unlisted band) stops the run if any aquifer cell holds that value.
* After writing, the closed file is audited: CF attributes, time axis strictly increasing, `time_bnds`, an image never stored twice, the
  frame and `roi_mask` equal to this run's polygon, `_FillValue` present, three strips of the first and last record outside the polygon
  equal to the fill; and records chosen for read-back are re-read from the closed file and compared.
* A file on Drive that was made with other settings (bands, window, `KEEP_WHOLE_WINDOW`), or whose records are not the first images of this
  run's list, stops the run and is never replaced.

**What to know before using the data**

* A month already in the file is not fetched again. Earth Engine re-issues images now and then (most of both collections were updated in
  April 2026); the file keeps each record's `source_update_time`, a run reports images that changed since, and `REPROCESS_CHANGED = True`
  writes the file again from the first changed record.
* `REBUILD = True` ignores the files on Drive and starts again from the first record.
* Earth Engine's MOD13A3 pixels were compared with NASA's HDF file for one tile and month on 2026-10-20 (identical); MCD12Q1's only through
  its attributes. The notebook was run locally against the real Earth Engine, not in Colab itself.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# earthengine-api, numpy, geopandas, shapely, pyproj and requests are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q netCDF4 rasterio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short rehearsal into .../_test/ ; False -> the whole record
TEST_RECORDS  = 4          # records (months of MOD13A3, years of MCD12Q1) in the rehearsal, spread over the whole record
TEST_FRAME_PX = None       # None -> the whole window; or e.g. 512 to rehearse on a 512 x 512 pixel square

DATASETS   = ["MOD13A3", "MCD12Q1"]   # one combined .nc file is made for each: MOD13A3 monthly 1 km, MCD12Q1 yearly 500 m
YEAR_START = None          # first year to export; None -> the first year Earth Engine has
YEAR_END   = None          # last year to export; None -> the latest year Earth Engine has (a later run adds new months)
KEEP_WHOLE_WINDOW = False  # False -> cells whose centre lies outside the aquifer polygon hold the fill value (roi_mask marks the
                           #          cells inside), and only chunks with aquifer cells take space in the file.
                           # True  -> every source cell of the window is kept (about 5 times more cells; the window is a
                           #          rectangle in MODIS's sinusoidal grid, four fifths of it is not aquifer)

PROJECT_ID = "msugw-503806"                                    # Earth Engine cloud project
EE_URL     = "https://earthengine-highvolume.googleapis.com"   # endpoint meant for many parallel requests
SKIP_IMAGES = []           # image ids to treat as permanently unavailable (see the stall message)

# ---- download ----
BLOCK_PX           = 2896  # a download block holds at most BLOCK_PX x BLOCK_PX pixels. Earth Engine counts 3 bytes
                           # per band value and refuses a request above 48 MiB; the limit for each image's number
                           # of bands is applied by itself, and a refused request is fetched in halves
N_DOWNLOAD_THREADS = 32    # blocks downloaded at a time (Earth Engine allows 40 concurrent requests)
REQUEST_DEADLINE_S = 180   # give up on a single request after this long (it is then retried)
STRIP_ROWS         = 512   # rows placed and written at a time (also the chunk height of the files)

COMPRESS_LEVEL = 2         # zlib level of the .nc files (lossless)
CHUNK_YX       = 512       # HDF5 chunk edge in y and x

MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
DRIVE_READ_TRIES  = 4      # attempts at reading a file's state from Drive before the run stops. The run never
                           # redoes or replaces a file because Drive failed to open it
MAX_RETRIES       = 4      # extra attempts per block
STALL_MINUTES     = 20     # stop cleanly after this long with no progress at all
BACKUP_INTERVAL_S = 3600   # copy the file in progress to Drive this often (and when the run ends or stops)
READBACK_RECORDS  = 4      # records per run re-read from the closed file and compared with what was written
                           # (every record in TEST_MODE)
CHECK_PX          = 64     # edge of a window compared with Earth Engine's own reading of the same pixels
CHECK_WINDOWS     = 4      # windows per compared image (placed on random aquifer cells)

REPROCESS_CHANGED = False  # True -> if Earth Engine has re-issued images that the file holds, write the file again from
                           #         the first changed record. False -> only report them (the stored values are kept)
REBUILD           = False  # True -> ignore the files on Drive and make the combined files again from the first record
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earth Engine sign-in, what each dataset and band is ═══════════════
import re, gc, io, sys, json, glob, time, math, zlib, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import shapely.ops
import requests
import rasterio.features
from affine import Affine
import ee

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a no-op there.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("modis")
for _n in ("urllib3.connectionpool", "googleapiclient", "google_auth_httplib2", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL  = "Ogallala"
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"
SCRATCH    = "/content/modis_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR    = f"{SCRATCH}/raw"                       # downloaded pixels of the record being written, then deleted
NC_DIR     = f"{SCRATCH}/nc"                        # the combined file being written
DISK_PATH  = "/content"                             # same filesystem as scratch AND Drive's local upload cache
LEDGER_NAME, LOG_FAILED, LOG_SKIPPED = "files_done.jsonl", "failed_images.jsonl", "skipped_images.jsonl"
for _d in (RAW_DIR, NC_DIR):
    os.makedirs(_d, exist_ok=True)

TIME_UNITS, TIME_CALENDAR = "days since 1970-01-01 00:00:00", "proleptic_gregorian"
EPOCH = datetime(1970, 1, 1)
FILE_VERSION = "v1"                                 # bump when the layout of the files changes

# MODIS's sinusoidal grid is a sphere of radius 6371007.181 m. Earth Engine labels it SR-ORG:6974 and its WKT text says
# "WGS 84", but its pixel positions follow the sphere (checked against ee.Image.pixelLonLat(): 0.00 m difference for the
# sphere, 17.9 km for the WGS 84 ellipsoid).
SINU_PROJ4 = "+proj=sinu +lon_0=0 +x_0=0 +y_0=0 +R=6371007.181 +units=m +no_defs"
CATALOGUE_URL = "https://storage.googleapis.com/earthengine-stac/catalog/MODIS/{tag}.json"


def crs_from_code(code):
    """pyproj CRS for an Earth Engine CRS code."""
    if str(code) == "SR-ORG:6974":
        return pyproj.CRS.from_proj4(SINU_PROJ4)
    return pyproj.CRS.from_user_input(code)


# ---------- the datasets: what each band is, and its no-data value ----------
# `fill` is the product's own no-data value. `fill_checked`: True where it was read from the attributes of NASA's own HDF
# file (MOD13A3.A2020183.h09v05.061, July 2020, and MCD12Q1.A2020001.h10v05.061, January 2020, read on 2026-10-22: fill,
# scale and valid range of every layer; the MOD13A3 scale factors and valid ranges had also been checked on 2026-10-20).
# Note that the four MOD13A3 reflectance bands have fill -1000, not the -3000 of the vegetation indices; the earlier
# version of this notebook declared -3000 for them.
# False where it is the value this project has used for the product (not read from NASA's file): no cell holding it is
# allowed inside the aquifer, and the run stops if one does, because then the fill and a real value could not be told apart.
# Stored values are NEVER scaled: `scale_factor` / `add_offset` are written as CF attributes so that readers decode them.
# Every band of a collection that is not listed here is stored as Earth Engine types it, with the largest value of its
# type as fill (and the same stop rule), and a warning.
def _vi(long_name, lo=-2000, hi=10000, fill=-3000, **kw):
    return dict(fill=fill, fill_checked=True, scale_factor=0.0001, valid_range=[lo, hi], long_name=long_name,
                units="1", **kw)


def _lc(long_name, lo, hi, **kw):
    return dict(fill=255, fill_checked=True, valid_range=[lo, hi], long_name=long_name, **kw)


def _angle(long_name, fill, lo, hi, **kw):
    return dict(fill=fill, fill_checked=True, scale_factor=0.01, valid_range=[lo, hi], long_name=long_name,
                units="degrees", **kw)


SPECS = {
    "MOD13A3": dict(
        collection="MODIS/061/MOD13A3", tag="MODIS_061_MOD13A3", step="month", pixel="1 km",
        out_name=f"MOD13A3_v061_1km_{ROI_LABEL}.nc",
        title="MODIS/Terra Vegetation Indices Monthly L3 Global 1 km (MOD13A3 version 6.1)",
        doi="10.5067/MODIS/MOD13A3.061",
        summary=("Monthly MOD13A3 version 6.1 (Terra) over the aquifer on MODIS's own 1 km sinusoidal grid, read from "
                 "Google Earth Engine as the stored integers. Nothing is scaled, resampled, masked by quality or filled."),
        bands={
            "NDVI": _vi("1 km monthly NDVI"),
            "EVI": _vi("1 km monthly EVI"),
            "sur_refl_b01": _vi("MODIS band 1 surface reflectance (red)", 0, 10000, fill=-1000),
            "sur_refl_b02": _vi("MODIS band 2 surface reflectance (NIR)", 0, 10000, fill=-1000),
            "sur_refl_b03": _vi("MODIS band 3 surface reflectance (blue)", 0, 10000, fill=-1000),
            "sur_refl_b07": _vi("MODIS band 7 surface reflectance (SWIR)", 0, 10000, fill=-1000),
            "DetailedQA": dict(
                fill=65535, fill_checked=True, long_name="VI quality indicators (bit field)",
                comment=("Bits 0-1 VI quality (0 good, 1 check other QA, 2 probably cloudy, 3 not produced); "
                         "bits 2-5 VI usefulness (0 highest ... 12 lowest, 13 not useful, 14 faulty L1B, 15 not processed); "
                         "bits 6-7 aerosol (0 climatology, 1 low, 2 intermediate, 3 high); bit 8 adjacent cloud; "
                         "bit 9 atmosphere BRDF correction; bit 10 mixed clouds; bits 11-13 land/water mask "
                         "(0 shallow ocean, 1 land, 2 coastline/shoreline, 3 shallow inland water, 4 ephemeral water, "
                         "5 deep inland water, 6 moderate/continental ocean, 7 deep ocean); bit 14 possible snow/ice; "
                         "bit 15 possible shadow. Source: Earth Engine catalogue MODIS/061/MOD13A3.")),
            "SummaryQA": dict(
                fill=-1, fill_checked=True, long_name="Quality reliability of VI pixel", units="1", valid_range=[0, 3],
                flags={0: "good_data", 1: "marginal_data", 2: "snow_ice", 3: "cloudy"}),
            "ViewZenith": _angle("View zenith angle of VI pixel", -10000, 0, 18000),
            "SolarZenith": _angle("Sun zenith angle of VI pixel", -10000, 0, 18000),
            "RelativeAzimuth": _angle(
                "Relative azimuth angle of VI pixel", -4000, -18000, 18000,
                comment=("NASA's fill value -4000 lies inside its own valid range, so a true -40.00 degree value "
                         "cannot be told from fill (the same in NASA's HDF file).")),
        }),
    "MCD12Q1": dict(
        collection="MODIS/061/MCD12Q1", tag="MODIS_061_MCD12Q1", step="year", pixel="500 m",
        out_name=f"MCD12Q1_v061_500m_{ROI_LABEL}.nc",
        title="MODIS/Terra+Aqua Land Cover Type Yearly L3 Global 500 m (MCD12Q1 version 6.1)",
        doi="10.5067/MODIS/MCD12Q1.061",
        summary=("Yearly MCD12Q1 version 6.1 (Terra and Aqua) over the aquifer on MODIS's own 500 m sinusoidal grid, read "
                 "from Google Earth Engine as the stored integers. Nothing is resampled, reclassified or filled."),
        classes_from_catalogue=True,
        bands={
            "LC_Type1": _lc("Land cover type 1: annual IGBP classification", 1, 17),
            "LC_Type2": _lc("Land cover type 2: annual UMD classification", 0, 15),
            "LC_Type3": _lc("Land cover type 3: annual LAI classification", 0, 10),
            "LC_Type4": _lc("Land cover type 4: annual BGC classification", 0, 8),
            "LC_Type5": _lc("Land cover type 5: annual plant functional types classification", 0, 11),
            "LC_Prop1": _lc("FAO-LCCS1 land cover layer", 1, 43),
            "LC_Prop2": _lc("FAO-LCCS2 land use layer", 1, 40),
            "LC_Prop3": _lc("FAO-LCCS3 surface hydrology layer", 1, 51),
            "LC_Prop1_Assessment": _lc("LCCS1 land cover layer confidence", 0, 100, units="percent"),
            "LC_Prop2_Assessment": _lc("LCCS2 land use layer confidence", 0, 100, units="percent"),
            "LC_Prop3_Assessment": _lc("LCCS3 surface hydrology layer confidence", 0, 100, units="percent"),
            "QC": _lc("Product quality flags", 0, 10),
            "LW": _lc("Binary land (2) / water (1) mask derived from MOD44W", 1, 2),
        }),
}
BAD_DATASETS = [d for d in DATASETS if d not in SPECS]
if BAD_DATASETS:
    raise ValueError(f"DATASETS holds {BAD_DATASETS}; the notebook knows {list(SPECS)}.")


class PipelineStop(RuntimeError):
    """Clean stop: everything on Drive is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class SceneUnavailable(Exception):
    """Earth Engine says the image does not exist. Nothing else may be mistaken for this: an image
    reported unavailable is left out of the record for good."""


class RunStopping(Exception):
    """Raised inside a download thread when the run is being stopped."""


class RequestTooLarge(Exception):
    """Earth Engine refused a pixel request for its size. Trying again cannot help; the rectangle
    is fetched as two halves instead."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_iso(s):
    s = s.rstrip("Z")
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%S.%f" if "." in s else "%Y-%m-%dT%H:%M:%S")


def ee_login():
    """Stored credentials first (an unattended restart needs no clicks); otherwise the sign-in flow."""
    try:
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    ee.data.setDeadline(REQUEST_DEADLINE_S * 1000)
    if ee.Number(1).add(1).getInfo() != 2:
        raise PipelineStop("Earth Engine did not answer a trivial request.")
    log.info("Earth Engine ready (project %s, high-volume endpoint)", PROJECT_ID)


log.info("datasets: %s | TEST_MODE=%s", " ".join(DATASETS), TEST_MODE)

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(folder, name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    os.makedirs(folder, exist_ok=True)
    with _LOG_LOCK:
        with open(os.path.join(folder, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: aquifer polygon, image list, the window on the product's own grid, block plan ═══════════════
_ROI_GDF = gpd.read_file(SHP_PATH)
if _ROI_GDF.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")
_POLY = {}


def roi_polygon(code=None):
    """The aquifer polygon in a given CRS (an Earth Engine CRS code; None -> the shapefile's own). The shapefile's
    parts are merged once; the merged outline is then reprojected, which is quick."""
    if None not in _POLY:
        g = _ROI_GDF.geometry
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        _POLY[None] = u if u.is_valid else u.buffer(0)
    if code not in _POLY:
        u = gpd.GeoSeries([_POLY[None]], crs=_ROI_GDF.crs).to_crs(crs_from_code(code)).iloc[0]
        _POLY[code] = u if u.is_valid else u.buffer(0)
    return _POLY[code]


log.info("Aquifer polygon: %.0f km2", roi_polygon("SR-ORG:6974").area / 1e6)


def with_retries(fn, what, tries=None, stop=None):
    """Run an Earth Engine request; retry with growing pauses. A missing asset is not retried.
    `stop` (a threading.Event) ends the waiting at once when the run is being stopped."""
    n = 1 + (MAX_RETRIES if tries is None else tries)
    last = None
    for attempt in range(n):
        if stop is not None and stop.is_set():
            raise RunStopping(what)
        try:
            return fn()
        except Exception as e:
            last = e
            msg = str(e)
            low = msg.lower()
            if "request size" in low and "must be less than" in low:
                raise RequestTooLarge(msg[:200])
            if "asset" in low and ("not found" in low or "does not exist" in low):
                raise SceneUnavailable(f"{what}: {msg[:200]}")
            if attempt + 1 < n:
                quota = "quota" in low or "429" in msg or "too many" in low
                pause = min((20 if quota else 2) * 2 ** attempt, 300)
                if stop is None:
                    time.sleep(pause)
                elif stop.wait(pause):
                    raise RunStopping(what)
    raise IOError(f"{what}: {type(last).__name__}: {str(last)[:300]}")


def span(nums):
    """[1, 2, 3, 7, 8] -> '1-3, 7-8'."""
    out, first = [], None
    for i, m in enumerate(nums):
        first = m if first is None else first
        if i + 1 == len(nums) or nums[i + 1] != m + 1:
            out.append(str(m) if first == m else f"{first}-{m}")
            first = None
    return ", ".join(out)


# ---------- the image list ----------
def ee_dtype(dt):
    """numpy type code of an Earth Engine band, from the catalogue entry's data type. A band whose range has no
    minimum is unsigned."""
    kind = str(dt.get("precision", "FLOAT")).upper()
    r = dt.get("range", {})
    lo, hi = r.get("min"), r.get("max")
    if kind == "INT" and hi is not None:
        if lo is None:
            return "u1" if hi <= 255 else "u2" if hi <= 65535 else "u4"
        for code, (a, b) in (("i1", (-128, 127)), ("i2", (-32768, 32767)), ("i4", (-2 ** 31, 2 ** 31 - 1))):
            if lo >= a and hi <= b:
                return code
        return "i8"
    return "f8" if kind == "DOUBLE" else "f4"


def _image_record(im):
    """One catalogue entry -> what the pipeline needs. Returns (record, None) or (None, reason)."""
    bands = im.get("bands", [])
    if not bands:
        return None, "no bands"
    g = bands[0]["grid"]
    if any(b["grid"] != g for b in bands):
        return None, "bands are not on one common grid"
    at = g["affineTransform"]
    if at.get("shearX", 0) or at.get("shearY", 0):
        return None, "rotated grid"
    return dict(id=im["id"], name=im["name"], t=im["startTime"], t_end=im.get("endTime", ""), update=im.get("updateTime", ""),
                bands=[b["id"] for b in bands], types={b["id"]: ee_dtype(b["dataType"]) for b in bands},
                crs=g["crsCode"], tx=float(at["translateX"]), ty=float(at["translateY"]), sx=float(at["scaleX"]),
                sy=float(at["scaleY"]), w=int(g["dimensions"]["width"]), h=int(g["dimensions"]["height"])), None


def list_images(spec):
    """Every image of the collection, in time order: (records, unusable entries)."""
    parent = "projects/earthengine-public/assets/" + spec["collection"]
    recs, odd, token = [], [], None
    while True:
        params = dict(parent=parent, pageSize=1000, view="FULL")
        if token:
            params["pageToken"] = token
        page = with_retries(lambda: ee.data.listImages(params), f"listing {spec['collection']}")
        for im in page.get("images", []):
            rec, why = _image_record(im)
            if rec is None:
                odd.append(dict(image=im.get("id", "?"), reason=why))
            else:
                recs.append(rec)
        token = page.get("nextPageToken")
        if not token:
            break
    recs.sort(key=lambda r: (r["t"], r["id"]))
    return recs, odd


# ---------- the window: the aquifer on the product's own pixel lattice ----------
# Every image of a MODIS collection lies on one lattice (cell size and origin of the global sinusoidal grid), but not
# every image has the same extent: MOD13A3 has images with 18000, 16800 and 15600 rows, cut at different latitudes. The
# window is cut once from the lattice of the first image: the rectangle of cells whose centre lies inside the aquifer
# polygon. It depends on geometry only, and every image is laid on it by a shift of whole cells, never resampled.
def on_lattice(r, ref):
    dc, dr = (r["tx"] - ref["tx"]) / ref["sx"], (r["ty"] - ref["ty"]) / ref["sy"]
    return (r["crs"] == ref["crs"] and r["sx"] == ref["sx"] and r["sy"] == ref["sy"]
            and abs(dc - round(dc)) < 1e-6 and abs(dr - round(dr)) < 1e-6)


def _rasterize(F):
    return rasterio.features.geometry_mask(
        [roi_polygon(F["crs"])], out_shape=(F["h"], F["w"]), invert=True, all_touched=False,       # cell centre inside the polygon
        transform=Affine(F["dx"], 0.0, F["x0"], 0.0, F["dy"], F["y0"]))


def make_frame(ref):
    """The window of the product of lattice `ref`: dict(crs, x0, y0 = top-left corner, dx, dy, w, h, n_roi), and
    the mask of the polygon on it. With TEST_FRAME_PX a small square around the aquifer cell nearest the middle."""
    crs, dx, dy = ref["crs"], ref["sx"], ref["sy"]
    left, bottom, right, top = roi_polygon(crs).bounds
    c0, c1 = math.floor((left - ref["tx"]) / dx), math.ceil((right - ref["tx"]) / dx)
    r0, r1 = math.floor((top - ref["ty"]) / dy), math.ceil((bottom - ref["ty"]) / dy)
    F = dict(crs=crs, x0=ref["tx"] + c0 * dx, y0=ref["ty"] + r0 * dy, dx=dx, dy=dy, w=c1 - c0, h=r1 - r0)
    mask = _rasterize(F)
    if not mask.any():
        raise PipelineStop("The aquifer polygon holds no cell centre of this product's grid.")
    rows, cols = np.flatnonzero(mask.any(axis=1)), np.flatnonzero(mask.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    if TEST_MODE and TEST_FRAME_PX:
        rr, cc = np.nonzero(mask[r0:r1:8, c0:c1:8])
        k = int(np.argmin((rr * 8 - (r1 - r0) / 2) ** 2 + (cc * 8 - (c1 - c0) / 2) ** 2))
        ra = max(r0, r0 + int(rr[k]) * 8 - TEST_FRAME_PX // 2)
        ca = max(c0, c0 + int(cc[k]) * 8 - TEST_FRAME_PX // 2)
        r0, r1, c0, c1 = ra, min(r1, ra + TEST_FRAME_PX), ca, min(c1, ca + TEST_FRAME_PX)
    sub = np.ascontiguousarray(mask[r0:r1, c0:c1])
    F = dict(crs=crs, x0=F["x0"] + c0 * dx, y0=F["y0"] + r0 * dy, dx=dx, dy=dy, w=c1 - c0, h=r1 - r0, n_roi=int(sub.sum()))
    return F, sub


def frame_id(F):
    return [F["crs"], F["x0"], F["y0"], F["dx"], F["dy"], F["w"], F["h"]]


def scene_window(r, F):
    """The part of image r that falls on frame F: dict(fr0, fc0 = its corner on the frame, h, w,
    sr0, sc0 = the same corner in the image's own pixel grid). {} when they do not overlap, None when
    the image is not on the frame's lattice (it is then left out, never resampled)."""
    dc, dr = (r["tx"] - F["x0"]) / F["dx"], (r["ty"] - F["y0"]) / F["dy"]
    if (r["crs"] != F["crs"] or r["sx"] != F["dx"] or r["sy"] != F["dy"]
            or abs(dc - round(dc)) > 1e-6 or abs(dr - round(dr)) > 1e-6):
        return None
    dc, dr = int(round(dc)), int(round(dr))
    fr0, fr1, fc0, fc1 = max(0, dr), min(F["h"], dr + r["h"]), max(0, dc), min(F["w"], dc + r["w"])
    if fr0 >= fr1 or fc0 >= fc1:
        return {}
    return dict(fr0=fr0, fc0=fc0, h=fr1 - fr0, w=fc1 - fc0, sr0=fr0 - dr, sc0=fc0 - dc)


# Earth Engine refuses a pixel request whose size, counted as pixels x bands x 3 bytes, is above 48 MiB
# (measured on 16-bit bands: 1448 x 1448 pixels of 8 bands are served, 1449 x 1448 are refused).
# Should the count differ for these bands, a refused request is fetched in halves instead of failing.
EE_REQUEST_BYTES   = 48 * 1024 ** 2
EE_BYTES_PER_VALUE = 3


def max_block_pixels(n_bands):
    """Most pixels one download block of n_bands bands may hold: BLOCK_PX, cut back to Earth Engine's limit."""
    return max(1, min(int(BLOCK_PX) ** 2, EE_REQUEST_BYTES // (n_bands * EE_BYTES_PER_VALUE)))


def tile_scene(h, w, max_px):
    """Cut an h x w pixel rectangle into few equal rectangles of at most max_px pixels each; returns
    (row edges, column edges). Rectangles rather than squares, because the time goes into the number of
    requests; long thin strips are avoided (side ratio at most 2) when that costs no more than a couple of
    requests, because compact blocks are more often skipped whole where the window leaves the aquifer."""
    cands = []
    for nr in range(1, h + 1):
        bh = -(-h // nr)                         # tallest block with nr rows
        if bh <= max_px:
            nc = -(-w // (max_px // bh))         # fewest columns that keep bh x width within max_px
            bw = -(-w // nc)
            cands.append((nr * nc, max(bh, bw) / min(bh, bw), nr, nc))
            if nc == 1 or nr > 2 * min(c[0] for c in cands):
                break
    fewest = min(c[0] for c in cands)
    compact = [c for c in cands if c[1] <= 2.0 and c[0] <= fewest + max(2, fewest // 8)]
    _, _, nr, nc = min(compact or [c for c in cands if c[0] == fewest])
    return [i * h // nr for i in range(nr + 1)], [j * w // nc for j in range(nc + 1)]


def plan_blocks(S, keep, n_bands):
    """Download blocks of an image window: rectangles (row0, col0, h, w), in the window's own pixel
    count, that hold at least one cell to keep."""
    rows, cols = tile_scene(S["h"], S["w"], max_block_pixels(n_bands))
    return [(a, c, b - a, d - c) for a, b in zip(rows, rows[1:]) for c, d in zip(cols, cols[1:])
            if keep[S["fr0"] + a:S["fr0"] + b, S["fc0"] + c:S["fc0"] + d].any()]


# ---------- what the catalogue says about the classes of a categorical band ----------
_CLASSES = {}


def catalogue_classes(spec):
    """{band: [(value, name, description)]} of the classified bands, from Earth Engine's public catalogue entry of the
    collection (the same text as the dataset page). {} when it cannot be read: the data are unaffected, only the flag
    attributes are then left out."""
    if spec["collection"] in _CLASSES:
        return _CLASSES[spec["collection"]]
    out = {}
    try:
        r = requests.get(CATALOGUE_URL.format(tag=spec["tag"]), timeout=60)
        r.raise_for_status()
        for b in r.json()["summaries"]["eo:bands"]:
            rows = []
            for c in b.get("gee:classes", []):
                text = " ".join(str(c["description"]).split())
                name = re.sub(r"[^a-z0-9]+", "_", text.split(":")[0].lower()).strip("_")
                rows.append((int(c["value"]), name, text))
            if rows:
                if len({n for _, n, _ in rows}) != len(rows):
                    rows = [(v, f"{n}_{v}", t) for v, n, t in rows]       # two classes of one name: keep them apart
                out[b["name"]] = rows
    except Exception as e:
        log.warning("The catalogue entry of %s could not be read (%s); flag attributes are left out.", spec["collection"], str(e)[:120])
    _CLASSES[spec["collection"]] = out
    return out


def band_table(spec, rec):
    """The variables of one dataset, in Earth Engine's band order: {band: dict(dtype, fill, fill_checked, attrs...)}.
    The type is Earth Engine's own; the fill must fit it."""
    cls = catalogue_classes(spec) if spec.get("classes_from_catalogue") else {}
    out, unlisted = {}, []
    for b in rec["bands"]:
        info = dict(spec["bands"].get(b, {}))
        dtype = rec["types"][b]
        if "fill" not in info:
            unlisted.append(b)
            kind = np.dtype(dtype)
            info.update(fill=(np.iinfo(kind).max if kind.kind == "u" else np.iinfo(kind).min) if kind.kind in "iu" else float("nan"),
                        fill_checked=False)
        if np.dtype(dtype).kind in "iu" and not np.iinfo(dtype).min <= info["fill"] <= np.iinfo(dtype).max:
            raise PipelineStop(f"{spec['collection']} {b}: the fill {info['fill']} does not fit Earth Engine's type {dtype}.")
        if b in cls:
            info["flag_values"] = np.array([v for v, _, _ in cls[b]], dtype=dtype)
            info["flag_meanings"] = " ".join(n for _, n, _ in cls[b])
            info["class_descriptions"] = " | ".join(f"{v}: {t}" for v, _, t in cls[b])
        elif "flags" in info:
            info["flag_values"] = np.array(list(info["flags"]), dtype=dtype)
            info["flag_meanings"] = " ".join(info["flags"].values())
        info["dtype"] = dtype
        out[b] = info
    if unlisted:
        log.warning("%s: bands %s are not described in this notebook. They are stored as Earth Engine types them, with %s as fill and "
                    "no scale, and the run stops if a cell inside the aquifer holds that value.", spec["collection"], unlisted,
                    "the largest value of the type")
    return out

In [ ]:
# ═══════════════ CELL 6: download the pixels of one record to scratch, check every block twice ═══════════════
def _grid(r, row0, col0, h, w):
    """A pixel rectangle of image r on the image's own grid: nothing is resampled on the way."""
    return dict(dimensions=dict(width=int(w), height=int(h)),
                affineTransform=dict(scaleX=r["sx"], shearX=0, translateX=r["tx"] + col0 * r["sx"],
                                     shearY=0, scaleY=r["sy"], translateY=r["ty"] + row0 * r["sy"]),
                crsCode=r["crs"])


def _check_block(arr, r, bands, shape):
    """A downloaded block must be exactly what was asked for: these bands, this shape, and for every band the
    integer type Earth Engine's catalogue lists for it."""
    if arr.dtype.names != tuple(bands):
        raise ValueError(f"bands {arr.dtype.names} instead of {tuple(bands)}")
    for b in bands:
        if arr.dtype[b] != np.dtype(r["types"][b]):
            raise ValueError(f"band {b}: data type {arr.dtype[b]} instead of the {np.dtype(r['types'][b])} the catalogue lists")
    if arr.shape != tuple(shape):
        raise ValueError(f"shape {arr.shape} instead of {tuple(shape)}")
    return arr


_SPLIT_NOTED = [False]


def fetch_array(r, bands, row0, col0, h, w, stop=None):
    """Earth Engine -> memory: one rectangle of image r as the stored integers (getPixels, so no Earth
    Engine computation is involved). Returns (array, bytes received). A rectangle that Earth Engine
    refuses for its size is fetched as two halves and put together again, so a wrong guess about its
    limit costs a few extra requests and never the run."""
    def once():
        raw = ee.data.getPixels(dict(assetId=r["name"], bandIds=list(bands), grid=_grid(r, row0, col0, h, w),
                                     fileFormat="NPY"))
        return _check_block(np.load(io.BytesIO(raw), allow_pickle=False), r, bands, (h, w)), len(raw)   # refuse a damaged reply

    try:
        return with_retries(once, f"{r['id']} block {row0},{col0}", stop=stop)
    except RequestTooLarge as e:
        if h * w < 4096:
            raise IOError(f"{r['id']}: Earth Engine refuses even a {h} x {w} pixel request ({e})")
        if not _SPLIT_NOTED[0]:
            _SPLIT_NOTED[0] = True
            log.warning("Earth Engine refused a %d x %d pixel request of %d bands for its size (%s). Such blocks "
                        "are fetched in halves from now on; lowering BLOCK_PX avoids the extra requests.",
                        h, w, len(bands), e)
        if h >= w:
            k = h // 2
            top, n1 = fetch_array(r, bands, row0, col0, k, w, stop)
            bottom, n2 = fetch_array(r, bands, row0 + k, col0, h - k, w, stop)
            return np.concatenate([top, bottom], axis=0), n1 + n2
        k = w // 2
        left, n1 = fetch_array(r, bands, row0, col0, h, k, stop)
        right, n2 = fetch_array(r, bands, row0, col0 + k, h, w - k, stop)
        return np.concatenate([left, right], axis=1), n1 + n2


def download_block(r, S, blk, mms, stop):
    """One block of one image: fetch its bands in one request and put them in the image's scratch
    files (one per band, in the band's own type). Returns the block with a checksum per band of what was written."""
    row0, col0, h, w = blk
    arr, nbytes = fetch_array(r, r["bands"], S["sr0"] + row0, S["sc0"] + col0, h, w, stop)
    if stop.is_set():                        # the scratch folder may already belong to the next run
        raise RunStopping(r["id"])
    crc = {}
    for b in r["bands"]:
        plane = np.ascontiguousarray(arr[b])
        crc[b] = zlib.crc32(plane)
        mms[b][row0:row0 + h, col0:col0 + w] = plane
    return dict(blk=blk, crc=crc, bytes=nbytes)


def verify_scene(paths, S, blocks, r):
    """Read an image's scratch files back and compare every block with the checksum taken on arrival."""
    for b in r["bands"]:
        mm = np.load(paths[b], mmap_mode="r")
        try:
            if mm.shape != (S["h"], S["w"]) or mm.dtype != np.dtype(r["types"][b]):
                raise IOError(f"{os.path.basename(paths[b])}: unexpected layout {mm.dtype} {mm.shape}")
            for blk in blocks:
                row0, col0, h, w = blk["blk"]
                if zlib.crc32(np.ascontiguousarray(mm[row0:row0 + h, col0:col0 + w])) != blk["crc"][b]:
                    raise IOError(f"{os.path.basename(paths[b])}: block {row0},{col0} read back differently from what arrived")
        finally:
            del mm


def download_image(tag, r, F, keep, fills, stop):
    """All blocks of one image, N_DOWNLOAD_THREADS at a time, each band into its own scratch file on the image's
    part of the frame. Returns (part, None) when every block arrived, passed its check and was read back from scratch
    unchanged; (None, reason) with a kind of failure otherwise. Cells that are not fetched (blocks without a cell to
    keep) are preset to the band's fill, so they can never be taken for data."""
    S = scene_window(r, F)
    if not S:
        return dict(S={}, paths={}, bytes=0, blocks=0), None
    blocks = plan_blocks(S, keep, len(r["bands"]))
    ensure_disk_free(f"{tag} download", extra_gb=S["h"] * S["w"] * sum(np.dtype(r["types"][b]).itemsize for b in r["bands"]) / 1024 ** 3)
    folder = os.path.join(RAW_DIR, tag)
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder, exist_ok=True)
    paths = {b: os.path.join(folder, f"{b}.npy") for b in r["bands"]}
    mms = {}
    for b in r["bands"]:
        mms[b] = np.lib.format.open_memmap(paths[b], mode="w+", dtype=r["types"][b], shape=(S["h"], S["w"]))
        mms[b][:] = fills[b]
    got, error = [], None
    ex = ThreadPoolExecutor(N_DOWNLOAD_THREADS)
    try:
        with timed("download"):
            futs = [ex.submit(download_block, r, S, blk, mms, stop) for blk in blocks]
            for fut in as_completed(futs):
                if stop.is_set():
                    raise RunStopping(tag)
                try:
                    got.append(fut.result())
                except SceneUnavailable as e:        # only Earth Engine saying "no such image" is permanent
                    error = error or ("permanent", str(e))
                except RunStopping:
                    raise
                except Exception as e:
                    error = error or ("transient", f"{type(e).__name__}: {e}")
    except BaseException:
        stop.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown(wait=True)
    for mm in mms.values():
        mm.flush()
    mms.clear()                                  # nothing may keep a scratch file open from here on
    futs = fut = None
    gc.collect()
    if error is None:
        try:
            with timed("download"):
                verify_scene(paths, S, got, r)
        except Exception as e:
            error = ("transient", f"{type(e).__name__}: {e}")
    if error is not None:
        shutil.rmtree(folder, ignore_errors=True)
        return None, error
    nbytes = sum(b["bytes"] for b in got)
    TOTALS["bytes"] = TOTALS.get("bytes", 0) + nbytes
    TOTALS["blocks"] = TOTALS.get("blocks", 0) + len(blocks)
    return dict(S=S, paths=paths, bytes=nbytes, blocks=len(blocks)), None


def fetch_record(tag, r, F, keep, fills, stop):
    """One image on scratch. A download that could not be completed is tried again with growing pauses; after
    STALL_MINUTES without any success the run stops cleanly. Returns the part, or None when Earth Engine says the
    image does not exist."""
    since, waits = None, 0
    while True:
        part, err = download_image(tag, r, F, keep, fills, stop)
        if part is not None:
            return part
        kind, why = err
        if kind == "permanent":
            log_events(DRIVE_DIR[0], LOG_FAILED, [dict(image=r["id"], reason=why)])
            log.warning("[%s] permanently unavailable: %s (%s)", tag, r["id"], why[:160])
            return None
        log.warning("[%s] %s could not be completed (%s)", tag, r["id"], why[:200])
        since = since or time.time()
        if time.time() - since > STALL_MINUTES * 60:
            raise PipelineStop(
                f"No progress for {STALL_MINUTES} min; stuck at {r['id']} ({why[:200]}). Everything on Drive is safe - "
                "re-run to resume. If this same image blocks every run while Earth Engine is otherwise fine, add its id "
                "to SKIP_IMAGES.")
        waits += 1
        if stop.wait(min(30 * 2 ** waits, 600)):
            raise RunStopping(tag)


DRIVE_DIR = [None]          # output folder of the dataset being run (set by run_dataset)


def fill_cells(r, F, keep, bands, fills, limit=3):
    """Up to `limit` aquifer cells (frame row, column) where image r holds the fill of some band, found by reading the
    whole window of one band after the other. These are the cells at which "no data" is stored, which random windows
    hit only by luck (about 1 cell in 40 000 in MOD13A3)."""
    S = scene_window(r, F)
    kept = keep[S["fr0"]:S["fr0"] + S["h"], S["fc0"]:S["fc0"] + S["w"]]
    out = []
    for band in bands:
        arr = fetch_array(r, [band], S["sr0"], S["sc0"], S["h"], S["w"])[0][band]
        at_fill = np.isnan(arr) if arr.dtype.kind == "f" and np.isnan(fills[band]) else arr == fills[band]
        hit = np.argwhere(at_fill & kept)
        if len(hit):
            a, c = hit[len(hit) // 2]
            out.append((int(a) + S["fr0"], int(c) + S["fc0"]))
        if len(out) >= limit:
            break
    return out


def compare_window(r, S, fr, fc, bands, fills, img):
    """One window centred on frame cell (fr, fc), read by the download and by Earth Engine's own evaluation."""
    n = min(CHECK_PX, S["h"], S["w"])
    row0 = int(min(max(fr - S["fr0"] - n // 2, 0), S["h"] - n)) + S["sr0"]
    col0 = int(min(max(fc - S["fc0"] - n // 2, 0), S["w"] - n)) + S["sc0"]
    mine, _ = fetch_array(r, bands, row0, col0, n, n)
    g = _grid(r, row0, col0, n, n)
    theirs = with_retries(lambda: ee.data.computePixels(dict(expression=img.select(bands).unmask(-99999).toInt32(), grid=g,
                                                             fileFormat="NUMPY_NDARRAY")), "cross-check")
    masked = same = 0
    for b in bands:
        a, t = mine[b].astype("i4"), theirs[b]
        has = t != -99999
        if not np.array_equal(a[has], t[has]):
            raise DataIntegrityError(f"{r['id']} band {b}: the downloaded pixels differ from Earth Engine's own reading")
        if np.count_nonzero(a[~has] != fills[b]):
            raise DataIntegrityError(f"{r['id']} band {b}: cells Earth Engine masks did not arrive as the fill {fills[b]} "
                                     f"(they hold {np.unique(a[~has])[:5].tolist()}); the notebook would store them as data")
        masked += int((~has).sum())
        same += int(has.sum())
    return same, masked


def cross_check(r, F, keep, bands, fills, random_windows=True):
    """Windows of image r read twice: by this notebook's download (stored integers on the image's own grid) and by
    Earth Engine's own evaluation of the image, with its mask. They must be identical where Earth Engine has a value,
    and where Earth Engine has none the download must hold the band's fill. This proves that the download path neither
    resamples nor shifts the pixels and that no-data arrives as the product's fill value. The windows sit on random
    aquifer cells (the same cells every run) and, because no-data is rare, on cells where the image holds the fill."""
    S = scene_window(r, F)
    if not S:
        raise DataIntegrityError(f"{r['id']} does not cover the window")
    img = ee.Image(r["name"].split("/assets/", 1)[1] if r["name"].startswith("projects/earthengine-public") else r["name"])
    cells = []
    if random_windows:
        rng = np.random.default_rng(20261022)
        pool = np.argwhere(keep[S["fr0"]:S["fr0"] + S["h"], S["fc0"]:S["fc0"] + S["w"]])
        cells = [(int(a) + S["fr0"], int(c) + S["fc0"]) for a, c in pool[rng.choice(len(pool), size=min(CHECK_WINDOWS, len(pool)), replace=False)]]
    on_fill = fill_cells(r, F, keep, bands, fills)
    same = masked = 0
    for fr, fc in cells + on_fill:
        s, m = compare_window(r, S, fr, fc, bands, fills, img)
        same, masked = same + s, masked + m
    if cells or on_fill:
        log.info("Cross-check OK on %s: %d windows (%d on cells holding a fill) of %d bands, %s values identical to Earth Engine's own "
                 "reading, %s masked cells arrived as the fill", r["id"], len(cells) + len(on_fill), len(on_fill), len(bands), f"{same:,}", f"{masked:,}")
    return dict(image=r["id"], windows=len(cells) + len(on_fill), fill_windows=len(on_fill), values=same, masked=masked)

In [ ]:
# ═══════════════ CELL 7: the combined CF NetCDF file: writer, state, read-back check, audit ═══════════════
STRIP = max(1, STRIP_ROWS // CHUNK_YX) * CHUNK_YX      # strips are whole rows of chunks, so each chunk is written once


def settings_signature(spec, F, table):
    """What a file is made from. A file with another signature is never added to."""
    return json.dumps(dict(collection=spec["collection"], file_version=FILE_VERSION, frame=frame_id(F),
                           whole_window=bool(KEEP_WHOLE_WINDOW),
                           bands=[[b, i["dtype"], str(i["fill"]), i.get("scale_factor")] for b, i in table.items()]), sort_keys=True)


def record_bounds(spec, t):
    """First instant of the month (or year) the image stands for, and the first instant after it. Earth Engine's own
    end time is 30 days (monthly) or 365 days (yearly) after the start for every image, which is not the end of the
    period, so the calendar period is used."""
    s = parse_iso(t)
    if spec["step"] == "month":
        return datetime(s.year, s.month, 1), datetime(s.year + (s.month == 12), s.month % 12 + 1, 1)
    return datetime(s.year, 1, 1), datetime(s.year + 1, 1, 1)


def _cf_attr(v):
    if isinstance(v, (str, int, float, np.number, np.ndarray)):
        return v
    if isinstance(v, (list, tuple)):
        return np.asarray(v)
    return str(v)


def create_nc(path, spec, F, table, polymask):
    """Create the empty CF file on the dataset's fixed frame. Written under a temporary name and renamed when complete."""
    crs = crs_from_code(F["crs"])
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    cy, cx = min(CHUNK_YX, F["h"]), min(CHUNK_YX, F["w"])
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("y", F["h"])
        ds.createDimension("x", F["w"])
        ds.createDimension("nv", 2)
        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(64,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name, tv.bounds = f"first day of the {spec['step']} (UTC)", "time_bnds"
        ds.createVariable("time_bnds", "f8", ("time", "nv"), chunksizes=(64, 2))
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis, yv.long_name = "projection_y_coordinate", "m", "Y", "northing of cell centre"
        yv[:] = F["y0"] + (np.arange(F["h"]) + 0.5) * F["dy"]
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis, xv.long_name = "projection_x_coordinate", "m", "X", "easting of cell centre"
        xv[:] = F["x0"] + (np.arange(F["w"]) + 0.5) * F["dx"]
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, _cf_attr(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{F['x0']} {F['dx']} 0.0 {F['y0']} 0.0 {F['dy']}"                 # GDAL order
        gm.long_name = f"MODIS sinusoidal grid (sphere of radius 6371007.181 m), cell {abs(F['dx']):.6f} m; Earth Engine code {F['crs']}"
        # 2-D latitude / longitude of the cell centres (CF auxiliary coordinates)
        X, Y = np.meshgrid(xv[:], yv[:])
        lon, lat = pyproj.Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(X, Y)
        for name, vals, std, unit in (("lat", lat, "latitude", "degrees_north"), ("lon", lon, "longitude", "degrees_east")):
            av = ds.createVariable(name, "f8", ("y", "x"), chunksizes=(cy, cx), **zkw)
            av.standard_name, av.units, av.long_name = std, unit, f"{std} of cell centre"
            av[:] = vals
        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=(cy, cx), **zkw)
        mv.long_name = f"1 where the cell centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values, mv.flag_meanings, mv.grid_mapping = np.array([0, 1], dtype="i1"), "outside_roi inside_roi", "crs"
        mv.coordinates = "lat lon"
        for r0 in range(0, F["h"], 2048):
            mv[r0:r0 + 2048, :] = polymask[r0:r0 + 2048].astype("i1")

        for b, info in table.items():
            dt, fill = info["dtype"], info["fill"]
            v = ds.createVariable(b, dt, ("time", "y", "x"), chunksizes=(1, cy, cx),
                                  fill_value=np.dtype(dt).type(fill), **zkw)
            v.grid_mapping, v.coordinates = "crs", "lat lon"
            v.source_collection, v.source_band = spec["collection"], b
            for k in ("long_name", "units", "comment", "class_descriptions"):
                if k in info:
                    setattr(v, k, info[k])
            if "scale_factor" in info:                       # CF attributes: the stored integers are NOT scaled
                v.scale_factor, v.add_offset = np.float64(info["scale_factor"]), np.float64(info.get("add_offset", 0.0))
            if "valid_range" in info:
                v.valid_range = np.array(info["valid_range"], dtype=dt)
            if "flag_values" in info:
                v.flag_values, v.flag_meanings = info["flag_values"], info["flag_meanings"]
            v.fill_note = ("_FillValue is the product's own no-data value, read from NASA's file." if info["fill_checked"] else
                           f"_FillValue {info['fill']} is the no-data value this project uses for the product; the run checked that no "
                           "cell inside the aquifer holds it as data.") + ("" if KEEP_WHOLE_WINDOW else " It also marks cells outside the polygon.")
        ds.createVariable("source_images", str, ("time",)).long_name = "Earth Engine image id the record was read from"
        ds.createVariable("source_update_time", str, ("time",)).long_name = (
            "Earth Engine's update time of that image when it was read (a later change means the image was re-issued)")

        ds.Conventions = "CF-1.9"
        ds.title = f"{spec['title']}, {ROI_LABEL} aquifer"
        ds.summary = spec["summary"] + (" Cells outside the aquifer polygon hold the fill value (roi_mask marks the cells inside)."
                                        if not KEEP_WHOLE_WINDOW else " Every cell of the window is kept (roi_mask marks the aquifer).")
        ds.institution = "NASA (source data), via Google Earth Engine; packaged by the MSU groundwater project"
        ds.source = f"Google Earth Engine collection {spec['collection']}"
        ds.references = f"https://doi.org/{spec['doi']}; https://developers.google.com/earth-engine/datasets/catalog/{spec['tag']}"
        ds.history = f"{iso(utcnow())} created by MODIS.ipynb"
        ds.date_created = iso(utcnow())
        ds.dataset, ds.roi, ds.step = spec["collection"], ROI_LABEL, spec["step"]
        ds.file_version = FILE_VERSION
        ds.variables_list = " ".join(table)
        ds.frame = json.dumps(dict(crs=F["crs"], x0=F["x0"], y0=F["y0"], dx=F["dx"], dy=F["dy"], w=F["w"], h=F["h"]))
        ds.settings_signature = settings_signature(spec, F, table)
        ds.processing_status = "in_progress"
        ds.committed_records = np.int32(0)
        ds.date_modified = iso(utcnow())
    finally:
        ds.close()
    os.replace(tmp, path)


def read_state(path, tries=1):
    """(state, None) of a combined file, or (None, why) when it cannot be used: unreadable, or a record list that
    disagrees with its last commit. From Drive it is tried several times: a Drive that is slow or only just mounted
    must never be mistaken for a damaged file."""
    why = None
    for attempt in range(max(1, tries)):
        if attempt:
            time.sleep(20 * attempt)
        try:
            with nc4.Dataset(path, "r") as ds:
                ds.set_auto_maskandscale(False)
                n = int(getattr(ds, "committed_records", -1))
                if n < 0 or n > len(ds.dimensions["time"]):
                    why = f"commits {n} records but holds {len(ds.dimensions['time'])} (an interrupted write)"
                else:
                    ids = [str(s) for s in ds.variables["source_images"][:n]]
                    upd = [str(s) for s in ds.variables["source_update_time"][:n]]
                    t = np.asarray(ds.variables["time"][:n], dtype="f8")
                    if n > 1 and not np.all(np.diff(t) > 0):
                        why = "has its time axis out of order"
                    else:
                        return dict(n=n, ids=ids, update=upd, status=str(getattr(ds, "processing_status", "in_progress")),
                                    signature=str(getattr(ds, "settings_signature", "")), frame=json.loads(str(getattr(ds, "frame", "{}"))),
                                    size=os.path.getsize(path)), None
        except Exception as e:
            why = f"cannot be read ({type(e).__name__}: {str(e)[:160]})"
        log.warning("%s %s (attempt %d of %d)", os.path.basename(path), why, attempt + 1, max(1, tries))
    return None, why


class Writer:
    """Single writer of the combined file on scratch. A record is written strip by strip and counts only when
    finish_record() has stored it in the file's own commit counter."""

    def __init__(self, path, table):
        self.path, self.table = path, table
        self.ds = nc4.Dataset(path, "a")
        self.ds.set_auto_maskandscale(False)
        self.n = int(self.ds.committed_records)

    def fresh(self):
        """True when record n is beyond everything the file holds. A record written over an older one (after a rewind)
        must write every strip, also those that are all fill, so that nothing of the old record stays."""
        return self.n >= len(self.ds.dimensions["time"])

    def write_strip(self, r0, r1, c0, c1, arrays, fresh):
        for name, a in arrays.items():
            if not fresh or np.count_nonzero(a != self.table[name]["fill"]):      # an array of fill values stays unwritten
                self.ds.variables[name][self.n, r0:r1, c0:c1] = a

    def finish_record(self, spec, r):
        v, i = self.ds.variables, self.n
        start, end = record_bounds(spec, r["t"])
        v["time"][i] = (start - EPOCH).days
        v["time_bnds"][i, :] = [(start - EPOCH).days, (end - EPOCH).days]
        v["source_images"][i] = r["id"]
        v["source_update_time"][i] = r["update"]
        self.n += 1
        self.commit()

    def commit(self, status=None):
        ds = self.ds
        ds.committed_records = np.int32(self.n)
        ds.date_modified = iso(utcnow())
        if status:
            ds.processing_status = status
        if self.n:
            ds.time_coverage_start = iso(EPOCH + timedelta(days=float(ds.variables["time"][0])))
            ds.time_coverage_end = iso(EPOCH + timedelta(days=float(ds.variables["time_bnds"][self.n - 1, 1])))
        ds.sync()

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def strip_windows(keep):
    """The parts of a frame that are written, as (r0, r1, c0, c1): strips of rows, their columns cut to the chunks
    that hold cells to keep. Rows without any such cell are left out, so that their chunks stay unwritten."""
    ny, nx = keep.shape
    for r0 in range(0, ny, STRIP):
        r1 = min(ny, r0 + STRIP)
        cols = np.flatnonzero(keep[r0:r1].any(axis=0))
        if cols.size:
            yield r0, r1, int(cols[0]) // CHUNK_YX * CHUNK_YX, min(nx, -(-(int(cols[-1]) + 1) // CHUNK_YX) * CHUNK_YX)


def place_strip(r, part, table, keep_strip, r0, r1, c0, c1):
    """Values of frame rows r0..r1, columns c0..c1 for every band, exactly as the product stores them. The image is laid on
    the frame by whole-cell shifts; nothing is computed, scaled or resampled. Cells outside the polygon (unless
    KEEP_WHOLE_WINDOW) and cells the image does not cover hold the fill value. Returns (arrays, counts of kept cells
    that hold the band's fill)."""
    h, w = r1 - r0, c1 - c0
    out = {b: np.full((h, w), table[b]["fill"], dtype=table[b]["dtype"]) for b in table}
    nfill = {b: 0 for b in table}
    S = part["S"]
    if not S:
        return out, nfill
    a0, a1 = max(r0, S["fr0"]), min(r1, S["fr0"] + S["h"])
    b0, b1 = max(c0, S["fc0"]), min(c1, S["fc0"] + S["w"])
    if a0 >= a1 or b0 >= b1:
        return out, nfill
    win = (slice(a0 - r0, a1 - r0), slice(b0 - c0, b1 - c0))
    inside = keep_strip[win]
    for b, info in table.items():
        mm = part["mm"][b]
        plane = np.asarray(mm[a0 - S["fr0"]:a1 - S["fr0"], b0 - S["fc0"]:b1 - S["fc0"]])
        at_fill = plane == info["fill"] if not (np.dtype(info["dtype"]).kind == "f" and np.isnan(info["fill"])) else np.isnan(plane)
        nfill[b] = int(np.count_nonzero(at_fill & inside))
        if nfill[b] and not info["fill_checked"]:
            raise DataIntegrityError(f"{r['id']} band {b}: {nfill[b]} cells inside the aquifer hold {info['fill']}, the value this "
                                     "file uses as fill. The notebook cannot tell them from missing data; nothing was copied "
                                     "to Drive. Choose another fill for this band in Cell 3 (SPECS).")
        out[b][win] = np.where(inside, plane, out[b][win])
    return out, nfill


def compose_record(r, part, writer, table, keep, keep_checksums):
    """One record onto the file, strip by strip. Returns (checksums or None, counters)."""
    opened = dict(part, mm={b: np.load(p, mmap_mode="r") for b, p in part["paths"].items()}) if part["S"] else part
    crcs = {} if keep_checksums else None
    stats = dict(kept=0, data=0, fill={b: 0 for b in table})
    main = next(iter(table))
    fresh = writer.fresh()
    try:
        for r0, r1, c0, c1 in strip_windows(keep):
            arrays, nfill = place_strip(r, opened, table, keep[r0:r1, c0:c1], r0, r1, c0, c1)
            writer.write_strip(r0, r1, c0, c1, arrays, fresh)
            stats["kept"] += int(np.count_nonzero(keep[r0:r1, c0:c1]))
            stats["data"] += int(np.count_nonzero(arrays[main] != table[main]["fill"]))
            for b, n in nfill.items():
                stats["fill"][b] += n
            if crcs is not None:
                for name, a in arrays.items():
                    crcs[(name, r0, r1, c0, c1)] = zlib.crc32(np.ascontiguousarray(a))
    finally:
        opened = None
    return crcs, stats


def readback_record(path, t, crcs):
    """One record read back from the closed file, strip by strip, and compared with the checksums taken when it was
    composed."""
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        for (name, r0, r1, c0, c1), want in crcs.items():
            got = np.ascontiguousarray(ds.variables[name][t, r0:r1, c0:c1])
            if zlib.crc32(got) != want:
                raise DataIntegrityError(f"{os.path.basename(path)} record {t} {name} rows {r0}-{r1}: the file "
                                         "differs from what was composed")
    return len(crcs)


def audit_nc(path, mask=None, F=None):
    """Read-only audit of a combined file. Returns (ok, problems, info). With the polygon mask, three strips of the
    first and last record are also checked: outside the polygon every cell must hold the fill."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            if not str(getattr(ds, "Conventions", "")).startswith("CF-"):
                problems.append("no CF Conventions attribute")
            names = str(getattr(ds, "variables_list", "")).split()
            need = ["time", "time_bnds", "y", "x", "lat", "lon", "crs", "roi_mask", "source_images", "source_update_time"] + names
            missing = [v for v in need if v not in ds.variables]
            if missing or not names:
                return False, [f"missing variables {missing}"], info
            n = int(getattr(ds, "committed_records", -1))
            if n < 0 or n > len(ds.dimensions["time"]):
                problems.append("the commit counter does not fit the time axis")
                n = 0
            t = np.asarray(ds.variables["time"][:n], dtype="f8")
            if n > 1 and not np.all(np.diff(t) > 0):
                problems.append("time not strictly increasing")
            tb = np.asarray(ds.variables["time_bnds"][:n], dtype="f8")
            if n and (not np.all(tb[:, 1] > tb[:, 0]) or not np.array_equal(tb[:, 0], t)):
                problems.append("time_bnds do not match the time axis")
            ids = [str(s) for s in ds.variables["source_images"][:n]]
            if len(set(ids)) != len(ids):
                problems.append("an image is stored twice")
            if F is not None and json.loads(str(ds.frame)) != dict(crs=F["crs"], x0=F["x0"], y0=F["y0"], dx=F["dx"], dy=F["dy"], w=F["w"], h=F["h"]):
                problems.append("the file is not on the frame of this run")
            for name in names:
                v = ds.variables[name]
                if v.dimensions != ("time", "y", "x") or getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{name}: unexpected dimensions or no grid_mapping")
                if "_FillValue" not in v.ncattrs():
                    problems.append(f"{name}: no _FillValue")
            ny, nx = len(ds.dimensions["y"]), len(ds.dimensions["x"])
            if mask is not None and n:
                if mask.shape != (ny, nx):
                    problems.append("the file is not on the frame of this run")
                elif not np.array_equal(np.asarray(ds.variables["roi_mask"][:]).astype(bool), mask):
                    problems.append("roi_mask differs from the polygon of this run")
                elif not KEEP_WHOLE_WINDOW:
                    for i in sorted({0, n - 1}):
                        for r0 in sorted({0, (ny // 2 // STRIP) * STRIP, ((ny - 1) // STRIP) * STRIP}):
                            m = mask[r0:r0 + STRIP]
                            for name in (names[0], names[-1]):
                                a = np.asarray(ds.variables[name][i, r0:r0 + STRIP, :])
                                if np.count_nonzero(a[~m] != ds.variables[name]._FillValue):
                                    problems.append(f"record {i} rows {r0}+: {name} has values outside the polygon")
            info.update(records=n, status=str(getattr(ds, "processing_status", "?")), shape=f"{ny}x{nx}", size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 8: record by record into one combined file per dataset; run ═══════════════
# How a run keeps its place. The file is built on scratch, one record (a month, or a year) at a time, and every record
# is read back from the file the moment it is committed. Every BACKUP_INTERVAL_S, and when the run ends or stops, the
# closed file is copied to Drive in one verified step (size and MD5 compared, then renamed over the previous copy).
# Nothing on Drive is ever half written. A runtime that dies loses at most the records since the last copy; the next
# run restores the Drive file, checks that it is the file this run would have made, and continues after its last
# committed record. Records already in the file are never fetched again (REPROCESS_CHANGED says what happens when Earth
# Engine has re-issued an image).
LAST = {}                       # the last file that reached Drive (for the message printed at a stop)


def stage_output(name):
    """Decide which copy of the file to continue from: scratch if it is at least as far along as Drive, otherwise
    Drive. Returns the local path (the file may not exist yet). A Drive file that cannot be read stops the run: it is
    never replaced because of that."""
    local, on_drive = os.path.join(NC_DIR, name), os.path.join(DRIVE_DIR[0], name)
    if REBUILD:
        if os.path.exists(local):
            os.remove(local)
        log.warning("REBUILD = True: %s starts again from the first record; the file on Drive is replaced at the first copy.", name)
        return local
    pl = read_state(local)[0] if os.path.exists(local) else None
    if os.path.exists(on_drive):
        pd_, why = read_state(on_drive, DRIVE_READ_TRIES)
        if pd_ is None:
            raise PipelineStop(f"{name} is on Drive but {why}. Nothing was changed or deleted. Run again in a few minutes (Drive can "
                               "be slow just after it is mounted). If the file really is damaged, move it out of the folder yourself and "
                               "run again.")
        if pl is None or pl["n"] < pd_["n"]:
            restore_from_drive(on_drive, local)
    elif os.path.exists(local) and pl is None:
        os.remove(local)                                  # unreadable scratch leftover, nothing on Drive
    return local


def backup(local, name):
    """Verified copy of the (closed) file to Drive; a Drive that will not take it stops the run cleanly."""
    try:
        robust_drive_copy(local, os.path.join(DRIVE_DIR[0], name))
    except Exception as e:
        raise PipelineStop(f"{name} could not be saved to Drive ({e}). Everything saved before it is still there - re-run to resume.")


class Progress:
    def __init__(self, total):
        self.total, self.done, self.t0 = total, 0, time.time()

    def add(self, n=1):
        self.done += n

    def line(self):
        rate = self.done / max(time.time() - self.t0, 1e-9)
        eta = (self.total - self.done) / rate / 60 if rate > 0 else float("nan")
        return f"{self.done}/{self.total} records ({100 * self.done / max(1, self.total):.0f} %), {rate * 60:.1f}/min, ETA {eta:.0f} min"


def pick_images(spec, images):
    """The images this run is to have in the file: the year range of the settings; in the rehearsal TEST_RECORDS of them,
    spread over the record."""
    sel = [r for r in images if (YEAR_START is None or int(r["t"][:4]) >= YEAR_START) and (YEAR_END is None or int(r["t"][:4]) <= YEAR_END)]
    if not sel:
        raise PipelineStop(f"{spec['collection']} has no image from YEAR_START ({YEAR_START}) to YEAR_END ({YEAR_END}); its record runs "
                           f"from {images[0]['t'][:10]} to {images[-1]['t'][:10]}.")
    if TEST_MODE:
        k = min(TEST_RECORDS, len(sel))
        sel = [sel[int((i + 0.5) * len(sel) / k)] for i in range(k)]
    return sel


def prepare(spec):
    """Everything a dataset needs before bulk work: images, frame, polygon mask, bands, fills."""
    imgs, odd = list_images(spec)
    log_events(DRIVE_DIR[0], LOG_SKIPPED, [dict(o, stage="image list") for o in odd])
    if not imgs:
        raise PipelineStop(f"{spec['collection']}: no usable image in Earth Engine's list.")
    ref = imgs[0]
    off = [r for r in imgs if not on_lattice(r, ref)]
    if off:
        log_events(DRIVE_DIR[0], LOG_SKIPPED, [dict(image=r["id"], stage="not on the grid of the first image") for r in off])
        log.warning("%d image(s) are not on the grid of the first image and are left out (never resampled): %s", len(off), [r["id"] for r in off][:5])
    imgs = [r for r in imgs if on_lattice(r, ref) and r["id"] not in SKIP_IMAGES]
    F, polymask = make_frame(ref)
    keep = np.ones_like(polymask) if KEEP_WHOLE_WINDOW else polymask
    out = []
    for r in imgs:
        S = scene_window(r, F)
        if not S:
            log_events(DRIVE_DIR[0], LOG_SKIPPED, [dict(image=r["id"], stage="does not reach the window")])
            continue
        out.append(r)
    imgs = out
    stamps = [r["t"] for r in imgs]
    if len(set(stamps)) != len(stamps):
        dup = sorted({t for t in stamps if stamps.count(t) > 1})
        raise PipelineStop(f"{spec['collection']} has more than one image for {dup[:3]}. The file holds one record per time; list the ids "
                           "to leave out in SKIP_IMAGES.")
    sets = {(tuple(r["bands"]), tuple(sorted(r["types"].items()))) for r in imgs}
    if len(sets) != 1:
        raise PipelineStop(f"{spec['collection']}: the images do not all have the same bands and types ({len(sets)} variants).")
    table = band_table(spec, imgs[0])
    fills = {b: np.dtype(i["dtype"]).type(i["fill"]) for b, i in table.items()}
    return imgs, F, polymask, keep, table, fills


def resume_plan(spec, local, sel, F, table):
    """(records to keep, state) of the file at `local`: 0 when there is none. A file made with other settings, or
    whose records are not the first ones of this run's image list, stops the run."""
    if not os.path.exists(local):
        return 0, None
    st, why = read_state(local)
    if st is None:
        raise PipelineStop(f"{os.path.basename(local)} cannot be used ({why}). Nothing was changed. Set REBUILD = True to make it again.")
    if st["signature"] != settings_signature(spec, F, table):
        raise PipelineStop(f"{os.path.basename(local)} was made with other settings (bands, grid window or KEEP_WHOLE_WINDOW). Restore "
                           "them in Cell 2, or set REBUILD = True.")
    if st["n"] > len(sel) or st["ids"] != [r["id"] for r in sel[:st["n"]]]:
        raise PipelineStop(f"The {st['n']} records in {os.path.basename(local)} are not the first {st['n']} images of this run "
                           f"(YEAR_START / YEAR_END changed, or Earth Engine added or removed an image inside the stored range). "
                           "Restore the year range, or set REBUILD = True.")
    changed = [i for i in range(st["n"]) if st["update"][i] != sel[i]["update"]]
    if changed:
        if REPROCESS_CHANGED:
            log.warning("Earth Engine has re-issued %d stored image(s), the first %s; writing again from record %d.", len(changed),
                        sel[changed[0]]["id"], changed[0])
            return changed[0], st
        log.warning("Earth Engine has re-issued %d image(s) that the file holds (the first: %s). Their stored values are kept; set "
                    "REPROCESS_CHANGED = True to write the file again from that record.", len(changed), sel[changed[0]]["id"])
    return st["n"], st


def run_dataset(name):
    spec = SPECS[name]
    DRIVE_DIR[0] = os.path.join(DRIVE_ROOT, name) + ("/_test" if TEST_MODE else "")
    os.makedirs(DRIVE_DIR[0], exist_ok=True)
    log.info("-" * 78)
    log.info("%s  (%s, %s): %s", name, spec["collection"], spec["pixel"], spec["out_name"])
    imgs, F, polymask, keep, table, fills = prepare(spec)
    sel = pick_images(spec, imgs)
    px = sum(np.dtype(i["dtype"]).itemsize for i in table.values())
    log.info("%d images in Earth Engine (%s to %s), %d for this run | window %d x %d cells of %.1f m, %s inside the aquifer | %d bands, "
             "%d bytes per cell", len(imgs), imgs[0]["t"][:10], imgs[-1]["t"][:10], len(sel), F["h"], F["w"], abs(F["dx"]), f"{F['n_roi']:,}",
             len(table), px)
    for b, i in table.items():
        log.info("   %-20s %-3s fill %-7s scale %-8s %s", b, i["dtype"], i["fill"], i.get("scale_factor", "-"),
                 "" if i["fill_checked"] else "(fill not read from NASA's file: no aquifer cell may hold it)")
    local = stage_output(spec["out_name"])
    keep_n, st = resume_plan(spec, local, sel, F, table)
    todo = sel[keep_n:]
    if not todo:
        log.info("%s on Drive already holds all %d records of this run - nothing to do.", spec["out_name"], keep_n)
        return dict(name=name, records=keep_n, new=0, path=os.path.join(DRIVE_DIR[0], spec["out_name"]))
    if keep_n:
        log.info("Continuing after record %d (%s): %d records to add", keep_n, sel[keep_n - 1]["id"], len(todo))
    ensure_disk_free(f"{name} file", extra_gb=2.0 * F["n_roi"] * px * len(todo) / 1024 ** 3)
    # ---- before any bulk work: the download must return exactly the pixels Earth Engine itself reads ----
    groups = {}
    for r in todo:
        groups.setdefault((r["tx"], r["ty"], r["w"], r["h"]), []).append(r)
    checks = []
    for gi, g in enumerate(list(groups.values())[:3]):
        first_group = gi == 0
        for k in range(min(3, len(g))):                      # up to 3 images spread over the record, until one holds no-data cells
            checks.append(cross_check(g[int((k + 0.5) * len(g) / min(3, len(g)))], F, keep, g[0]["bands"], fills, random_windows=(k == 0 and first_group)))
            if checks[-1]["fill_windows"]:
                break
        else:
            log.warning("None of the compared images of this grid holds a no-data cell inside the aquifer: the fill values were checked "
                        "against random windows only.")
    log_events(DRIVE_DIR[0], "cross_check.jsonl", checks)
    log.info("PREFLIGHT OK (%d image grid(s) compared with Earth Engine)", len(groups))
    # ---- the file ----
    if not os.path.exists(local):
        create_nc(local, spec, F, table, polymask)
    writer = Writer(local, table)
    if st is not None and keep_n < st["n"]:
        writer.n = keep_n
        writer.commit("in_progress")
    check_set = set(range(len(todo))) if TEST_MODE else set(np.unique(np.linspace(0, len(todo) - 1, READBACK_RECORDS).astype(int)).tolist())
    checks_crc, counts = {}, dict(failed=0, no_data=0)
    stop, pool = threading.Event(), ThreadPoolExecutor(1)
    prog = Progress(len(todo))
    t_run, last_backup, n0, audit_failed = time.time(), time.time(), writer.n, False
    fetch = lambda r: fetch_record(f"{name}_{r['t'][:7]}", r, F, keep, fills, stop)
    try:
        try:
            fut = pool.submit(fetch, todo[0])
            for i, r in enumerate(todo):
                part = fut.result()
                fut = pool.submit(fetch, todo[i + 1]) if i + 1 < len(todo) else None
                if part is None:
                    counts["failed"] += 1
                    log.warning("[%s] %s is not in the file (permanently unavailable)", name, r["id"])
                    continue
                t_c, t = time.time(), writer.n
                crcs, stats = compose_record(r, part, writer, table, keep, True)
                writer.ds.sync()
                with timed("read-back"):
                    for (b, r0, r1, c0, c1), want in crcs.items():            # through the open file, before the record counts
                        if zlib.crc32(np.ascontiguousarray(writer.ds.variables[b][t, r0:r1, c0:c1])) != want:
                            raise DataIntegrityError(f"{name} record {t} {b} rows {r0}-{r1}: the file differs from what was composed")
                writer.finish_record(spec, r)
                if i in check_set:
                    checks_crc[t] = crcs
                TIMES["compose + write"] = TIMES.get("compose + write", 0.0) + time.time() - t_c
                shutil.rmtree(os.path.join(RAW_DIR, f"{name}_{r['t'][:7]}"), ignore_errors=True)
                prog.add()
                share = 100 * stats["data"] / max(1, stats["kept"])
                bad = {b: n for b, n in stats["fill"].items() if n}
                log.info("[%s] %s: %.2f %% of the kept cells hold %s data | %s | %s downloaded | free %.0f GB", name, r["t"][:10], share,
                         next(iter(table)), prog.line(), human(TOTALS.get("bytes", 0)), free_gb())
                if bad:
                    log.debug("   cells holding the fill, per band: %s", bad)
                gc.collect()
                if time.time() - last_backup >= BACKUP_INTERVAL_S and i + 1 < len(todo):
                    writer.close()
                    backup(local, spec["out_name"])
                    writer = Writer(local, table)
                    last_backup = time.time()
        except BaseException:
            stop.set()                           # the record being fetched in the background stops too
            pool.shutdown(wait=False, cancel_futures=True)
            raise
        pool.shutdown(wait=True)
        last_image = sel[-1]["id"] == imgs[-1]["id"] and not TEST_MODE and YEAR_END is None
        writer.commit("test" if TEST_MODE else ("complete" if last_image else "partial"))
        writer.close()
        # ---------- the closed file is checked before it is the copy on Drive ----------
        t_v = time.time()
        compared = sum(readback_record(local, t, c) for t, c in checks_crc.items())
        good, problems, info = audit_nc(local, polymask, F)
        if not good:
            audit_failed = True
            raise DataIntegrityError(f"{spec['out_name']} failed its audit and was not copied to Drive: {problems}")
        TIMES["checks"] = TIMES.get("checks", 0.0) + time.time() - t_v
        backup(local, spec["out_name"])
        LAST.update(file=spec["out_name"])
        size = os.path.getsize(local)
        log_events(DRIVE_DIR[0], LEDGER_NAME, [dict(file=spec["out_name"], records=info["records"], new=len(todo) - counts["failed"],
                                                    size=size, status=info["status"], seconds=round(time.time() - t_run))])
        TOTALS["files"] = TOTALS.get("files", 0) + 1
        TOTALS["file_bytes"] = TOTALS.get("file_bytes", 0) + size
        log.info("[%s] DONE: %d records (%d new), %s on Drive, %d arrays read back from the closed file and compared, %.1f min", name,
                 info["records"], len(todo) - counts["failed"], human(size), compared, (time.time() - t_run) / 60)
        return dict(name=name, records=info["records"], new=len(todo) - counts["failed"], size=size, seconds=time.time() - t_run,
                    path=os.path.join(DRIVE_DIR[0], spec["out_name"]), total_records=len(imgs), per_record=size / max(1, info["records"]))
    finally:
        try:
            if writer.ds is not None:
                writer.close()
        except Exception:
            pass
        shutil.rmtree(RAW_DIR, ignore_errors=True)
        os.makedirs(RAW_DIR, exist_ok=True)
        if os.path.exists(local) and writer.n > n0 and not audit_failed and LAST.get("file") != spec["out_name"]:
            try:                                    # a stopped run still saves the records it has committed
                log.warning("Saving the %d committed records to Drive.", writer.n)
                st2, _ = read_state(local)
                if st2 is not None:
                    robust_drive_copy(local, os.path.join(DRIVE_DIR[0], spec["out_name"]))
            except Exception as e:
                log.error("The committed records could not be saved to Drive: %s", e)


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    TOTALS.clear()
    LAST.clear()
    log.info("=" * 78)
    log.info("  MODIS %s | one combined file per dataset | TEST_MODE=%s", " + ".join(DATASETS), TEST_MODE)
    log.info("=" * 78)
    results, stopped = [], None
    try:
        for d in (RAW_DIR, NC_DIR):              # downloads left by an earlier run are never used
            if d == RAW_DIR:
                shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        ee_login()
        ensure_disk_free("start")
        local, remote = os.path.join(NC_DIR, "_write_test.txt"), os.path.join(DRIVE_ROOT, "_write_test.txt")
        with open(local, "w") as f:
            f.write(iso(utcnow()))
        robust_drive_copy(local, remote)         # Drive must be writable before any bulk work
        os.remove(remote)
        os.remove(local)
        for d in DATASETS:
            results.append(run_dataset(d))
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."

    log.info("=" * 78)
    if stopped is not None:
        where = (f"\n  The last file that reached Drive is {LAST['file']}." if LAST.get("file") else
                 "\n  No file was finished in this run; the next run continues from what Drive holds.")
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}{where}\n  Give Drive a "
              "minute to finish uploading before you disconnect the runtime.\n" + "=" * 78 + "\n")
    log.info("  this run: %d file(s) copied to Drive (%s), %s downloaded in %d blocks, %.1f min", TOTALS.get("files", 0),
             human(TOTALS.get("file_bytes", 0)), human(TOTALS.get("bytes", 0)), TOTALS.get("blocks", 0), (time.time() - t0) / 60)
    if TIMES:
        total_t = sum(TIMES.values())
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s ({100 * v / total_t:.0f} %)" for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    if TEST_MODE:
        for r in results:
            if r.get("seconds") and r.get("new"):
                per = r["seconds"] / r["new"]
                log.info("  PROJECTION %s (from %d records of this rehearsal, rough): %d records in all = ~%.1f h of runtime (%.1f s per record) and "
                         "~%.1f GB on Drive (%s per record)", r["name"], r["new"], r["total_records"], r["total_records"] * per / 3600, per,
                         r["total_records"] * r["per_record"] / 1024 ** 3, human(r["per_record"]))
    log.info("=" * 78)
    return results


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 9: what is in the files ═══════════════
DESCRIBE_RECORDS = 6       # records per variable that are read for the statistics (spread over the record; all in TEST_MODE)


def describe(name):
    spec = SPECS[name]
    path = os.path.join(DRIVE_ROOT, name) + ("/_test" if TEST_MODE else "") + "/" + spec["out_name"]
    if not os.path.exists(path):
        print(f"{name}: no file at {path}")
        return
    ok, problems, info = audit_nc(path)
    with nc4.Dataset(path) as ds:
        ds.set_auto_maskandscale(False)
        n = info["records"]
        t = np.asarray(ds["time"][:n], dtype="f8")
        days = [EPOCH + timedelta(days=float(v)) for v in t]
        inside = np.asarray(ds["roi_mask"][:]).astype(bool)
        print(f"{path}\n{human(os.path.getsize(path))} | {n} records {days[0]:%Y-%m-%d} .. {days[-1]:%Y-%m-%d} | {len(ds.dimensions['y'])} x "
              f"{len(ds.dimensions['x'])} cells | {int(inside.sum()):,} inside the aquifer | status {info['status']} | "
              f"{'passes the audit' if ok else 'AUDIT PROBLEMS: ' + '; '.join(problems)}")
        steps = np.diff([d.year * 12 + d.month for d in days]) if spec["step"] == "month" else np.diff([d.year for d in days])
        gaps = [f"{days[i]:%Y-%m} -> {days[i + 1]:%Y-%m}" for i in np.flatnonzero(steps != 1)]
        print(f"records missing between: {gaps if gaps else 'none'}"
              + (" (the rehearsal holds records spread over the record, so gaps are expected)" if TEST_MODE else "") + "\n")
        pick = list(range(n)) if TEST_MODE else sorted(set(np.linspace(0, n - 1, min(n, DESCRIBE_RECORDS)).astype(int).tolist()))
        print(f"cells inside the aquifer in {len(pick)} sampled records:")
        print(f"{'variable':20s} {'type':6s} {'fill':>7s} {'valid_range':>16s} {'with data':>10s} {'at fill':>9s} {'outside range':>14s}   data range (stored integers)")
        for b in str(ds.variables_list).split():
            v = ds[b]
            fill, vr = v._FillValue, (v.valid_range.tolist() if "valid_range" in v.ncattrs() else None)
            n_data = n_fill = n_out = 0
            lo, hi = None, None
            for i in pick:
                a = np.asarray(v[i])[inside]
                at_fill = a == fill
                data = a[~at_fill]
                n_data += data.size
                n_fill += int(at_fill.sum())
                if data.size:
                    lo = data.min() if lo is None else min(lo, data.min())
                    hi = data.max() if hi is None else max(hi, data.max())
                    if vr:
                        n_out += int(np.count_nonzero((data < vr[0]) | (data > vr[1])))
            tot = max(1, n_data + n_fill)
            rng = f"{lo} .. {hi}" if lo is not None else "-"
            print(f"{b:20s} {str(v.dtype):6s} {fill:>7} {str(vr) if vr else '-':>16s} {100 * n_data / tot:9.2f}% {100 * n_fill / tot:8.2f}% "
                  f"{n_out:>14,d}   {rng}")
        print()


for _d in DATASETS:
    describe(_d)

# Reading the files (xarray applies scale_factor and turns the fill value and cells outside valid_range into NaN):
#   import xarray as xr
#   root = "/content/drive/MyDrive/MSUGWB"
#   ndvi = xr.open_dataset(f"{root}/MOD13A3/MOD13A3_v061_1km_Ogallala.nc", chunks={"time": 1, "y": 512, "x": 512}).NDVI
#   lc   = xr.open_dataset(f"{root}/MCD12Q1/MCD12Q1_v061_500m_Ogallala.nc", chunks={"time": 1}).LC_Type1
# The grid is MODIS's own sinusoidal grid; `lat` and `lon` hold the 2-D coordinates of the cell centres, `roi_mask` is 1 inside
# the aquifer. To get the stored integers instead (for example to test a quality bit of DetailedQA), open with
# mask_and_scale=False. MOD13A3 and MCD12Q1 are on different grids (1 km and 500 m): they are separate files by design.